In [ ]:
import pandas as pd

oportunidad = pd.read_excel("../data/inputs/oportunidad.xlsx")
oportunidad.head()

## Contexto: estrategia FDS anterior

Resultado de la estrategia de precios del fin de semana anterior (jun 2026), usado como referencia para la nueva estrategia.

In [ ]:
analisis_path = "../data/utils/analisis_estrategia_pricing_fds_jun2026.xlsx"

detalle_skus = pd.read_excel(analisis_path, sheet_name="Detalle SKUs")
resumen_departamento = pd.read_excel(analisis_path, sheet_name="Resumen Departamento")
top_uplift = pd.read_excel(analisis_path, sheet_name="Top Uplift")
sin_venta_fds = pd.read_excel(analisis_path, sheet_name="Sin Venta FDS")

detalle_skus.head()

## Estrategia de ofertas FDS

Construye la tabla de SKUs priorizados con mecánica recomendada y margen simulado post-oferta, con margen mínimo del 22%.

In [ ]:
MARGEN_MIN = 22.0

# Limpiar oportunidad: quitar filas de resumen/filtros al final del export y parsear SKU
uni = oportunidad.dropna(subset=["STORE_ID"]).copy()
uni = uni[uni["SKU + Nombre"].astype(str).str.match(r"^\d+\s*-\s*.+")].copy()

uni[["SKU", "Nombre"]] = uni["SKU + Nombre"].str.split(" - ", n=1, expand=True)
uni["SKU"] = uni["SKU"].astype(int)
uni["STORE_ID"] = uni["STORE_ID"].astype(int)
uni.head()

### Excluir Black list y descuentos comerciales vigentes

`Descuentos comerciales.xlsx` trae dos pestañas por SKU + Tienda (`ItemId` + `AreaId`): `Black list` (exclusion fija) y `DEscuentos comerciales` (calendario de campañas, `FechaInicio`/`FechaFin`). Se excluye del universo cualquier SKU en Black list, o con un descuento comercial cuyo rango de fechas se cruce con el fin de semana objetivo (viernes 3 - domingo 5 jul 2026), para no duplicar mecánicas sobre un SKU que ya tiene oferta comprometida.

In [ ]:
WEEKEND_INICIO = pd.Timestamp("2026-07-03")
WEEKEND_FIN = pd.Timestamp("2026-07-05")

descuentos_path = "../data/inputs/Descuentos comerciales.xlsx"

black_list = pd.read_excel(descuentos_path, sheet_name="Black list")
black_pares = set(zip(black_list["ItemId"].astype(int), black_list["AreaId"].astype(int)))

comerciales = pd.read_excel(descuentos_path, sheet_name="DEscuentos comerciales")
comerciales["ItemId"] = pd.to_numeric(comerciales["ItemId"], errors="coerce")
comerciales["AreaId"] = pd.to_numeric(comerciales["AreaId"], errors="coerce")
comerciales = comerciales.dropna(subset=["ItemId", "AreaId", "FechaInicio", "FechaFin"])
comerciales["ItemId"] = comerciales["ItemId"].astype(int)
comerciales["AreaId"] = comerciales["AreaId"].astype(int)

vigentes = comerciales[
    (comerciales["FechaInicio"] <= WEEKEND_FIN) & (comerciales["FechaFin"] >= WEEKEND_INICIO)
]
comerciales_pares = set(zip(vigentes["ItemId"], vigentes["AreaId"]))

excluir_pares = black_pares | comerciales_pares
print(f"Pares (SKU, Tienda) en Black list: {len(black_pares)}")
print(f"Pares (SKU, Tienda) con descuento comercial vigente ese finde: {len(comerciales_pares)}")

antes = len(uni)
mask_excluir = uni.set_index(["SKU", "STORE_ID"]).index.isin(excluir_pares)
uni = uni[~mask_excluir].copy()
print(f"Excluidos por Black list / descuento comercial vigente: {antes - len(uni)}")
print(f"Universo restante: {len(uni)}")

## Catalogo SKU / Departamento / Categoria (Snowflake)

Reemplaza el cruce contra `analisis_estrategia_pricing_fds_jun2026.xlsx` (solo 110 SKUs, 3.5% de cobertura) por el catalogo real. Requiere `.env` con `SNOWFLAKE_USER`, `SNOWFLAKE_WAREHOUSE` y `SNOWFLAKE_ROLE` llenos. Autenticacion SSO de Google: al ejecutar la celda se abre el navegador.

In [ ]:
from snowflake_conn import get_connection

conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE()")
print(cur.fetchone())

In [ ]:
cur.execute("""
    SELECT DISTINCT SKU, DEPARTMENT, CATEGORY
    FROM MX_JUSTO_PROD.SANDBOX.FULL_MASTER_CATALOG
""")
columnas = [c[0] for c in cur.description]
catalogo = pd.DataFrame(cur.fetchall(), columns=columnas)
catalogo = catalogo.rename(columns={"DEPARTMENT": "Departamento", "CATEGORY": "Categoria"})

# SKU puede venir como VARCHAR sucio; forzar a entero y descartar lo no numerico
catalogo["SKU"] = pd.to_numeric(catalogo["SKU"], errors="coerce")
catalogo = catalogo.dropna(subset=["SKU"])
catalogo["SKU"] = catalogo["SKU"].astype(int)

# Un SKU podria tener mas de una combinacion Departamento/Categoria (data quality) - nos quedamos con la primera
dup = catalogo["SKU"].duplicated().sum()
print(f"SKUs con mas de una combinacion Departamento/Categoria: {dup}")
catalogo = catalogo.drop_duplicates(subset="SKU", keep="first")

print(f"SKUs en catalogo: {len(catalogo)}")
catalogo.head()

In [ ]:
uni = uni.merge(catalogo, on="SKU", how="left")

cobertura = uni["Departamento"].notna().sum()
print(f"Cobertura Departamento/Categoria: {cobertura}/{len(uni)} filas ({cobertura / len(uni) * 100:.1f}%)")
uni.head()

In [ ]:
# Validar que MARGEN = margen sobre precio neto de IVA e IEPS
# (PRECIO JUSTO viene con impuestos incluidos)
precio_neto_check = uni["PRECIO JUSTO"] / (1 + uni["IVA"] / 100) / (1 + uni["IEPS"] / 100)
margen_check = (precio_neto_check - uni["COSTO"]) / precio_neto_check * 100
diff = (margen_check - uni["MARGEN"]).abs()
print(f"Diferencia vs columna MARGEN: media={diff.mean():.4f} pts, "
      f"filas con diff>0.5pts={(diff > 0.5).sum()}/{len(uni)}")

In [ ]:
# Filtros de exclusion: margen base < 22%, o inelastico/sin datos de elasticidad
antes = len(uni)
elegibles = uni[uni["MARGEN"] >= MARGEN_MIN]
excl_margen = antes - len(elegibles)

antes = len(elegibles)
elegibles = elegibles[~elegibles["TAG ELAS"].isin(["INELASTICO", "SIN DATOS"])].copy()
excl_elas = antes - len(elegibles)

print(f"Excluidos por margen base < {MARGEN_MIN}%: {excl_margen}")
print(f"Excluidos por TAG_ELAS inelastico/sin datos: {excl_elas}")
print(f"SKUs elegibles: {len(elegibles)}")

In [ ]:
TOLERANCIA_CASCADA = 2.0  # puntos % max. que se sacrifican para usar una mecanica con nombre


def simular_oferta(precio, costo, ieps, iva):
    """Calcula el descuento maximo exacto (d_max) que deja el margen (neto de
    IVA/IEPS) justo en MARGEN_MIN - esa es la unica fuente de verdad, nunca se
    ofrece menos descuento del que el margen permite.

    Se usa una mecanica con nombre (2x1/3x2/4x3/5x4/6x5, la familia "paga N-1
    lleva N") solo si su descuento real esta a <= TOLERANCIA_CASCADA puntos de
    d_max (para no sacrificar atractivo solo por usar un nombre redondo). Si
    ninguna mecanica con nombre esta lo bastante cerca, se usa el descuento
    exacto, presentado segun la "regla del 100" de pricing: SPON (%) para
    precios < $100 (el % se percibe mas grande), BNSDP ($) para precios >=
    $100 (el monto en pesos pega mas fuerte).
    """

    def margen_neto(precio_final):
        precio_neto = precio_final / (1 + iva / 100) / (1 + ieps / 100)
        return (precio_neto - costo) / precio_neto * 100

    precio_neto_min = costo / (1 - MARGEN_MIN / 100)
    precio_final_min = precio_neto_min * (1 + iva / 100) * (1 + ieps / 100)
    d_max = max((1 - precio_final_min / precio) * 100, 0)

    if d_max <= 0:
        return "Sin oferta viable", precio, margen_neto(precio), "Sin oferta viable", round(d_max, 2)

    for nombre, ratio in [("2x1", 50.0), ("3x2", 100 / 3), ("4x3", 25.0), ("5x4", 20.0), ("6x5", 100 / 6)]:
        if ratio <= d_max:
            if (d_max - ratio) <= TOLERANCIA_CASCADA:
                pe = precio * (1 - ratio / 100)
                return nombre, pe, margen_neto(pe), "Cascada (paga N-1 lleva N)", round(d_max, 2)
            break  # esta mecanica deja mas de 2 pts sin usar; ninguna mas chica sera mejor

    # Descuento exacto, redondeado a multiplos de 5 (o al entero si el espacio es <5)
    d = (int(d_max) // 5) * 5 if d_max >= 5 else int(d_max)
    d = max(d, 0)
    pe = precio * (1 - d / 100)
    margen = margen_neto(pe)

    if precio >= 100:
        ahorro = round(precio - pe)
        return f"BNSDP ahorra ${ahorro}", pe, margen, "BNSDP (pesos, precio >= $100)", round(d_max, 2)
    return f"SPON {d}%", pe, margen, "SPON (%, precio < $100)", round(d_max, 2)


resultados = elegibles.apply(
    lambda r: simular_oferta(r["PRECIO JUSTO"], r["COSTO"], r["IEPS"], r["IVA"]), axis=1
)
elegibles["Mecanica recomendada"] = resultados.str[0]
elegibles["Precio oferta efectivo"] = resultados.str[1].round(2)
elegibles["Margen simulado post-oferta %"] = resultados.str[2].round(2)
elegibles["Tipo mecanica"] = resultados.str[3]
elegibles["Descuento maximo posible %"] = resultados.str[4]

antes = len(elegibles)
elegibles = elegibles[elegibles["Mecanica recomendada"] != "Sin oferta viable"]
print(f"Excluidos por no encontrar oferta viable: {antes - len(elegibles)}")

## Validacion de demanda por historico real

`simular_oferta()` (arriba) solo responde "¿aguanta el margen?". No dice si la mecanica va a mover volumen. Esta seccion agrega, sin tocar la logica de `simular_oferta()`:
- Elasticidad y demanda base por SKU desde `ATHENEA`.
- Historial real de promos desde `DISCOUNT_CAMPAIGN_EVENT`.
- Ventas historicas FDS de los SKUs elegibles desde `MASTER_ORDERLINE`.
- Proyeccion de demanda esperada aplicando elasticidad a la mecanica ya asignada.
- Un nivel de confianza por SKU (Alta/Media/Baja) segun que tan respaldada esta esa proyeccion.
- Un backtesting: comparar el uplift que la elasticidad habria proyectado en promos pasadas contra el uplift que realmente se vendio, para saber si Athenea sobreestima o subestima.

Usa la conexion `conn`/`cur` ya abierta arriba - no abre una nueva. `MASTER_ORDERLINE` tiene registros con timestamps corruptos fuera de rango real, por eso todas las queries filtran `DATETIME_DELIVERY` explicitamente, y `PRODUCT_ID` se castea a `VARCHAR` para evitar el error de encoding conocido.

### Chequeo rapido de esquema

No puedo ejecutar Snowflake yo mismo (requiere tu SSO), asi que no tengo forma de confirmar en vivo que columnas como `AVG_UNITS_PER_DAY`, `AVG_UNITS_PER_DAY_WITH_ELAST` o `STATUS_SAP` existen tal cual en `ATHENEA`, ni que `BY_BULK_STRATEGY_DISCOUNT_TYPE`/`BY_BULK_STRATEGY_VALUE` siguen igual en `DISCOUNT_CAMPAIGN_EVENT`. Esta celda trae 3 filas de cada tabla para verlas antes de correr las queries grandes - si algo no calza, es mas facil corregirlo aqui que despues de una query de minutos sobre `MASTER_ORDERLINE`.

In [ ]:
cur.execute("SELECT * FROM MX_JUSTO_PROD.DR_GENERAL.ATHENEA LIMIT 3")
print("ATHENEA columnas:", [c[0] for c in cur.description])
print(cur.fetchall())
print()

cur.execute("SELECT * FROM MX_JUSTO_PROD.ODS_MS_PRICING_V.DISCOUNT_CAMPAIGN_EVENT LIMIT 3")
print("DISCOUNT_CAMPAIGN_EVENT columnas:", [c[0] for c in cur.description])
print(cur.fetchall())

### Paso A - Elasticidad y demanda base (Athenea)

Trae elasticidad y unidades/dia promedio por SKU+Tienda desde `ATHENEA`, y hace merge con `elegibles` (nuevas columnas, no pisa nada existente). Como la query solo trae SKUs con `ELASTICITY_BY_SKU IS NOT NULL`, algunos SKUs de `elegibles` van a quedar sin elasticidad propia despues del merge - para esos se calcula un promedio de elasticidad por Categoria (usando los SKUs que si tienen dato) como fallback, y se marca con `Elasticidad es fallback` para que el Paso E sepa que es una proyeccion menos confiable.

In [ ]:
cur.execute("""
    SELECT
        PRODUCT_ID::VARCHAR  AS SKU,
        WAREHOUSE_ID         AS STORE_ID,
        ELASTICITY_BY_SKU,
        AVG_UNITS_PER_DAY,
        AVG_UNITS_PER_DAY_WITH_ELAST
    FROM MX_JUSTO_PROD.DR_GENERAL.ATHENEA
    WHERE WAREHOUSE_ID IN (9, 14)
      AND STATUS_SAP = 'Prendido'
      AND ELASTICITY_BY_SKU IS NOT NULL
""")
columnas = [c[0] for c in cur.description]
athenea = pd.DataFrame(cur.fetchall(), columns=columnas)
athenea["SKU"] = pd.to_numeric(athenea["SKU"], errors="coerce")
athenea = athenea.dropna(subset=["SKU"])
athenea["SKU"] = athenea["SKU"].astype(int)
athenea["STORE_ID"] = athenea["STORE_ID"].astype(int)
athenea = athenea.drop_duplicates(subset=["SKU", "STORE_ID"], keep="first")

if "ELASTICITY_BY_SKU" in elegibles.columns:
    print("AVISO: 'elegibles' ya tenia columnas de Athenea, se van a sobreescribir en este merge")

elegibles = elegibles.merge(athenea, on=["SKU", "STORE_ID"], how="left")

con_elasticidad_propia = elegibles["ELASTICITY_BY_SKU"].notna().sum()
sin_elasticidad_propia = elegibles["ELASTICITY_BY_SKU"].isna().sum()
print(f"SKUs con elasticidad propia (por SKU+Tienda): {con_elasticidad_propia}/{len(elegibles)}")
print(f"SKUs sin elasticidad propia (necesitan fallback por Categoria): {sin_elasticidad_propia}/{len(elegibles)}")

# Fallback: promedio de elasticidad por Categoria, calculado solo con SKUs que si tienen dato propio
elasticidad_por_categoria = (
    elegibles.dropna(subset=["ELASTICITY_BY_SKU"])
    .groupby("Categoria")["ELASTICITY_BY_SKU"]
    .mean()
)

elegibles["Elasticidad es fallback"] = elegibles["ELASTICITY_BY_SKU"].isna()
elegibles["ELASTICITY_BY_SKU"] = elegibles.apply(
    lambda r: elasticidad_por_categoria.get(r["Categoria"], float("nan"))
    if pd.isna(r["ELASTICITY_BY_SKU"]) else r["ELASTICITY_BY_SKU"],
    axis=1,
).astype(float)
elegibles["AVG_UNITS_PER_DAY"] = elegibles["AVG_UNITS_PER_DAY"].fillna(0)

cubiertos_con_fallback = elegibles["ELASTICITY_BY_SKU"].notna().sum()
print(f"SKUs con elasticidad (propia + fallback por categoria): {cubiertos_con_fallback}/{len(elegibles)}")
print(f"SKUs sin ninguna elasticidad (ni propia ni de categoria): {elegibles['ELASTICITY_BY_SKU'].isna().sum()}")
print(f"✅ Paso A - Athenea: {con_elasticidad_propia} con elasticidad propia, "
      f"{cubiertos_con_fallback - con_elasticidad_propia} con fallback por categoria")

### Paso B - Historial real de promos (DISCOUNT_CAMPAIGN_EVENT)

Trae el calendario de promos pasadas desde enero 2025, para los SKUs del universo elegible. Se usa en el Paso E (confianza) y en el Paso F (backtesting) para saber que SKUs ya tuvieron una promo real y comparar contra lo que paso en ventas.

In [ ]:
cur.execute("""
    SELECT
        REGEXP_REPLACE(PRODUCT_ID, '[^0-9]', '') AS SKU,
        WAREHOUSE_ID                              AS STORE_ID,
        START_DATE,
        END_DATE,
        BY_BULK_STRATEGY_DISCOUNT_TYPE            AS TIPO_MECANICA,
        BY_BULK_STRATEGY_VALUE                    AS VALOR_DESCUENTO
    FROM MX_JUSTO_PROD.ODS_MS_PRICING_V.DISCOUNT_CAMPAIGN_EVENT
    WHERE START_DATE >= '2025-01-01'
""")
columnas = [c[0] for c in cur.description]
promos_historicas = pd.DataFrame(cur.fetchall(), columns=columnas)
promos_historicas["SKU"] = pd.to_numeric(promos_historicas["SKU"], errors="coerce")
promos_historicas = promos_historicas.dropna(subset=["SKU", "STORE_ID", "START_DATE", "END_DATE"])
promos_historicas["SKU"] = promos_historicas["SKU"].astype(int)
promos_historicas["STORE_ID"] = promos_historicas["STORE_ID"].astype(int)

skus_elegibles = set(elegibles["SKU"])
promos_relevantes = promos_historicas[promos_historicas["SKU"].isin(skus_elegibles)].copy()

skus_con_historial_promo = set(zip(promos_relevantes["SKU"], promos_relevantes["STORE_ID"]))
print(f"Filas de promos historicas (todo el catalogo): {len(promos_historicas)}")
print(f"Filas de promos historicas de SKUs en el universo elegible: {len(promos_relevantes)}")
print(f"Pares (SKU, Tienda) elegibles con al menos una promo pasada: {len(skus_con_historial_promo)}")
print(f"✅ Paso B - Historial de promos: {len(skus_con_historial_promo)} pares (SKU, Tienda) con historial")

### Paso C - Ventas historicas de los SKUs elegibles (MASTER_ORDERLINE)

Trae ventas diarias reales de 2025-01-01 a 2026-10-22 para los SKUs elegibles (no todo el catalogo), usadas en el Paso F para el backtesting. El SKU se parametriza en el `IN (...)` en vez de concatenarse como string. Se agrega `LIMIT 1000000` como proteccion preventiva - son ~1100 SKUs x 2 tiendas x ~22 meses de historial, no tengo forma de medir el tiempo de esta query sin ejecutarla, asi que documento el limite aqui en vez de esperar a que tarde. Si corre rapido y no se llega al limite, se puede quitar.

In [ ]:
skus_param = tuple(int(s) for s in elegibles["SKU"].unique())
placeholders = ",".join(["%s"] * len(skus_param))

query_ventas = f"""
    SELECT
        PRODUCT_ID::VARCHAR                                    AS SKU,
        STORE_ID,
        TO_DATE(DATETIME_DELIVERY)                             AS FECHA,
        DAYOFWEEK(DATETIME_DELIVERY)                           AS DIA_SEMANA,
        SUM(QUANTITY_FULFILLED_PZ)                             AS UNIDADES,
        SUM(AMOUNT_GROSS_DELIVERED)                            AS GMV
    FROM MX_JUSTO_PROD.DR_MASTER_TABLES.MASTER_ORDERLINE
    WHERE STATUS_ORDER            = 'delivered'
      AND STORE_ID                IN (9, 14)
      AND QUANTITY_FULFILLED_PZ   > 0
      AND DATETIME_DELIVERY       BETWEEN '2025-01-01' AND '2026-10-22'
      AND PRODUCT_ID::VARCHAR     IN ({placeholders})
    GROUP BY 1, 2, 3, 4
    LIMIT 1000000
"""
cur.execute(query_ventas, skus_param)
columnas = [c[0] for c in cur.description]
ventas_historicas = pd.DataFrame(cur.fetchall(), columns=columnas)
ventas_historicas["SKU"] = pd.to_numeric(ventas_historicas["SKU"], errors="coerce")
ventas_historicas = ventas_historicas.dropna(subset=["SKU"])
ventas_historicas["SKU"] = ventas_historicas["SKU"].astype(int)
ventas_historicas["STORE_ID"] = ventas_historicas["STORE_ID"].astype(int)
ventas_historicas["FECHA"] = pd.to_datetime(ventas_historicas["FECHA"])

print(f"Filas de ventas historicas: {len(ventas_historicas)}")
print(f"SKUs con al menos un dia de venta en el historico: {ventas_historicas['SKU'].nunique()}/{len(skus_param)}")
if len(ventas_historicas) >= 1_000_000:
    print("AVISO: se llego al LIMIT de 1,000,000 filas - el historico esta truncado, considerar acotar fechas")
print(f"✅ Paso C - Ventas historicas: {len(ventas_historicas)} filas, "
      f"{ventas_historicas['SKU'].nunique()} SKUs con venta registrada")

### Paso D - Proyeccion de demanda con la mecanica ya asignada

Para `Descuento efectivo %` no uso la funcion de regex sobre el texto de la mecanica que venia en el pedido original: esa funcion tiene un bug real - para mecanicas BNSDP (ej. `"BNSDP ahorra $25"`), el regex `r"(\d+)"` extrae el **25 en pesos** y lo trata como si fuera 25%, lo cual esta mal (mezcla unidades). En vez de reconstruir el % desde el texto, lo calculo directo desde los dos precios que ya tenemos (`PRECIO JUSTO` vs `Precio oferta efectivo`) - es exacto y no depende de parsear texto.

Con eso aplico la formula de elasticidad que diste para proyectar unidades/dia esperadas.

In [ ]:
def proyectar_demanda(avg_units_per_day, elasticidad, descuento_pct):
    """
    Demanda esperada con promo.
    elasticidad es negativa (ej: -2.07), descuento_pct es positivo (ej: 10.0).
    Formula: demanda_base x (1 + |elasticidad| x (descuento / 100))
    """
    uplift = abs(elasticidad) * (descuento_pct / 100)
    return avg_units_per_day * (1 + uplift)


elegibles["Descuento efectivo %"] = (
    (elegibles["PRECIO JUSTO"] - elegibles["Precio oferta efectivo"]) / elegibles["PRECIO JUSTO"] * 100
).round(2)

elegibles["Demanda base (u/día)"] = elegibles["AVG_UNITS_PER_DAY"].round(2)

elegibles["Demanda proyectada FDS (u/día)"] = elegibles.apply(
    lambda r: proyectar_demanda(r["AVG_UNITS_PER_DAY"], r["ELASTICITY_BY_SKU"], r["Descuento efectivo %"])
    if pd.notna(r["ELASTICITY_BY_SKU"]) else float("nan"),
    axis=1,
).round(2)

elegibles["Uplift proyectado %"] = (
    (elegibles["Demanda proyectada FDS (u/día)"] - elegibles["Demanda base (u/día)"])
    / elegibles["Demanda base (u/día)"].replace(0, float("nan")) * 100
).round(1)

print(f"SKUs con proyeccion de demanda calculada: {elegibles['Demanda proyectada FDS (u/día)'].notna().sum()}/{len(elegibles)}")
print(f"Uplift proyectado promedio: {elegibles['Uplift proyectado %'].mean():.1f}%")
print(f"✅ Paso D - Proyeccion de demanda: uplift promedio {elegibles['Uplift proyectado %'].mean():.1f}%")

### Paso E - Nivel de confianza de la proyeccion

- **Alta**: elasticidad propia (no fallback) + historial real de promo para ese SKU+Tienda en `DISCOUNT_CAMPAIGN_EVENT`.
- **Media**: elasticidad propia, pero sin historial de promo (nunca se probo en la practica).
- **Baja**: se uso la elasticidad de fallback por Categoria (`Elasticidad es fallback` = True), o no hay elasticidad en absoluto.

In [ ]:
def nivel_confianza(row):
    if pd.isna(row["ELASTICITY_BY_SKU"]):
        return "Baja"
    if row["Elasticidad es fallback"]:
        return "Baja"
    tiene_historial = (row["SKU"], row["STORE_ID"]) in skus_con_historial_promo
    return "Alta" if tiene_historial else "Media"


elegibles["Confianza proyección"] = elegibles.apply(nivel_confianza, axis=1)

print("Distribucion de confianza:")
print(elegibles["Confianza proyección"].value_counts())
print(f"✅ Paso E - Confianza: {(elegibles['Confianza proyección'] == 'Alta').sum()} Alta, "
      f"{(elegibles['Confianza proyección'] == 'Media').sum()} Media, "
      f"{(elegibles['Confianza proyección'] == 'Baja').sum()} Baja")

### Paso F - Backtesting: uplift real vs proyectado por elasticidad

Diagnostico, no cambia la seleccion de SKUs. Para cada SKU+Tienda con historial de promo (Paso B), comparo unidades/dia vendidas **durante** esa promo vs **fuera** de ella en `ventas_historicas` (Paso C), y calculo el uplift real observado. Lo comparo contra el uplift que la elasticidad de Athenea habria proyectado para ese mismo % de descuento historico.

Limitaciones que quiero que sepas: solo uso promos de tipo `PERCENTAGE` (para `FIXED` el valor viene en pesos, no es directamente comparable sin mas trabajo), y solo SKUs con al menos 2 dias de venta durante la promo y 5 dias fuera de ella, para que el promedio no sea ruido.

In [ ]:
promos_pct = promos_relevantes[
    (promos_relevantes["TIPO_MECANICA"] == "PERCENTAGE") & promos_relevantes["VALOR_DESCUENTO"].notna()
]

elasticidad_por_sku_tienda = elegibles.set_index(["SKU", "STORE_ID"])["ELASTICITY_BY_SKU"].to_dict()

filas_backtest = []
for _, promo in promos_pct.iterrows():
    sku, store = promo["SKU"], promo["STORE_ID"]
    elasticidad = elasticidad_por_sku_tienda.get((sku, store))
    if elasticidad is None or pd.isna(elasticidad):
        continue

    ventas_sku = ventas_historicas[(ventas_historicas["SKU"] == sku) & (ventas_historicas["STORE_ID"] == store)]
    if ventas_sku.empty:
        continue

    en_promo = ventas_sku[
        (ventas_sku["FECHA"] >= promo["START_DATE"]) & (ventas_sku["FECHA"] <= promo["END_DATE"])
    ]
    fuera_promo = ventas_sku[
        (ventas_sku["FECHA"] < promo["START_DATE"]) | (ventas_sku["FECHA"] > promo["END_DATE"])
    ]
    if len(en_promo) < 2 or len(fuera_promo) < 5:
        continue

    unidades_promo = en_promo["UNIDADES"].mean()
    unidades_base = fuera_promo["UNIDADES"].mean()
    if unidades_base <= 0:
        continue

    uplift_real = (unidades_promo - unidades_base) / unidades_base * 100
    uplift_proyectado = abs(elasticidad) * float(promo["VALOR_DESCUENTO"])

    filas_backtest.append({
        "SKU": sku,
        "STORE_ID": store,
        "Uplift real %": round(uplift_real, 1),
        "Uplift proyectado %": round(uplift_proyectado, 1),
        "Error (proyectado - real)": round(uplift_proyectado - uplift_real, 1),
    })

backtest = pd.DataFrame(filas_backtest)

if len(backtest):
    mae = backtest["Error (proyectado - real)"].abs().mean()
    bias = backtest["Error (proyectado - real)"].mean()
    direccion = "sobreestima" if bias > 0 else "subestima"
    print(f"SKUs+Tienda evaluados en backtesting: {len(backtest)}")
    print(f"MAE: {mae:.1f} puntos porcentuales")
    print(f"Sesgo (bias): {bias:+.1f} pts")
    print(f"CONCLUSION: la elasticidad de Athenea {direccion} el uplift real en {abs(bias):.1f}% en promedio "
          f"(MAE {mae:.1f} pts, n={len(backtest)})")
else:
    mae = bias = None
    print("No hay suficientes pares SKU+Tienda con historial de promo % y ventas antes/durante para backtesting")

print(f"✅ Paso F - Backtesting: n={len(backtest)}, "
      f"MAE={mae:.1f}pts" if mae is not None else f"✅ Paso F - Backtesting: sin datos suficientes")

### Deteccion "Tema Mundial"

No hay merchandising con licencia FIFA en este catalogo (0 matches para "mundial", "fútbol", "FIFA", "selección"), asi que se usa un proxy: categorias tipicas de "ver el partido" (cerveza, refrescos, botanas saladas, salsas picantes/dip). Revisado a mano para evitar falsos positivos (ej. se descarto "cacahuate" porque en este catalogo solo matcheaba crema de cacahuate/yogurt, no cacahuates como botana).

In [ ]:
import re

PATRON_MUNDIAL = re.compile(
    r"\b(?:"
    r"cerveza|refresco|"
    r"papas?\s+(?:fritas?|pringles|ruffles)|totopos?|nachos?|chicharr\w*|"
    r"palomitas|botanas?|"
    r"habanero|valentina|cholula|clamato|b[uú]falo"
    r")\b",
    re.IGNORECASE,
)
elegibles["Tema Mundial"] = elegibles["Nombre"].str.contains(PATRON_MUNDIAL, na=False)
print(f"SKUs marcados como afines al Mundial: {elegibles['Tema Mundial'].sum()}")

In [ ]:
# Dia sugerido: domingo para despensa (cobertura parcial de Departamento),
# viernes para alta rotacion, sabado como dia insignia por defecto ("mayor impacto")
def dia_sugerido(row):
    if row["Departamento"] == "Despensa":
        return "Domingo"
    if row["TAG_ROTACION"] == "Alta rotacion":
        return "Viernes"
    return "Sabado"


elegibles["Dia sugerido"] = elegibles.apply(dia_sugerido, axis=1)

# Orden: TAG_ELAS (mejor primero), luego TAG_ROTACION, luego margen simulado descendente
orden_elas = {"ALTAMENTE ELASTICO": 0, "ELASTICO": 1, "POCO ELASTICO": 2}
orden_rot = {"Alta rotacion": 0, "Rotacion normal": 1, "Baja rotacion": 2, "Sin rotacion": 3}

elegibles["_orden_elas"] = elegibles["TAG ELAS"].map(orden_elas)
elegibles["_orden_rot"] = elegibles["TAG_ROTACION"].map(orden_rot)

elegibles = elegibles.sort_values(
    by=["_orden_elas", "_orden_rot", "Margen simulado post-oferta %"],
    ascending=[True, True, False],
)

estrategia_fds = elegibles.rename(
    columns={"MARGEN": "Margen base %", "TAG ELAS": "TAG_ELAS"}
)[
    [
        "STORE_ID", "SKU", "Nombre", "Departamento", "Categoria",
        "Margen base %", "TAG_ELAS", "TAG_ROTACION",
        "Tipo mecanica", "Mecanica recomendada", "Descuento maximo posible %",
        "Margen simulado post-oferta %", "Precio oferta efectivo",
        "Descuento efectivo %", "Demanda base (u/día)", "Demanda proyectada FDS (u/día)",
        "Uplift proyectado %", "Confianza proyección",
        "Dia sugerido", "Tema Mundial",
    ]
]

estrategia_fds.to_excel("../data/output/estrategia_fds.xlsx", index=False)

print("=== RESUMEN ===")
print(f"Total SKUs incluidos: {len(estrategia_fds)}")
print()
print("Distribucion por tipo de mecanica:")
print(estrategia_fds["Tipo mecanica"].value_counts())
print()
print("Distribucion por mecanica:")
print(estrategia_fds["Mecanica recomendada"].value_counts())
print()
print("Distribucion por dia:")
print(estrategia_fds["Dia sugerido"].value_counts())
print()
print(f"SKUs Tema Mundial: {estrategia_fds['Tema Mundial'].sum()}")
print(estrategia_fds[estrategia_fds["Tema Mundial"]]["Dia sugerido"].value_counts())
print()
print("=== RESUMEN EXTENDIDO - VALIDACION DE DEMANDA ===")
print("Distribucion por confianza de proyeccion:")
print(estrategia_fds["Confianza proyección"].value_counts())
print()
if len(backtest):
    print(f"Backtesting: n={len(backtest)}, MAE={mae:.1f}pts, sesgo={bias:+.1f}pts")
else:
    print("Backtesting: sin datos suficientes para calcular MAE/sesgo")
print()
print("Uplift proyectado promedio por dia sugerido:")
print(estrategia_fds.groupby("Dia sugerido")["Uplift proyectado %"].mean().round(1))